In [0]:
from pyspark.sql.functions import *
silver_df = (
    spark.readStream
    .format("delta")
    .load(
        "abfss://silver@travelappprojectstorage.dfs.core.windows.net/tourist_location_silver"
    )
)

In [0]:
active_df = silver_df.filter(
    col("activity_status") == "ACTIVE"
)


In [0]:
missing_df = silver_df.filter(
    col("inactivity_minutes") > 15
)

In [0]:
active_query = (
    active_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/gold_active_tourists"
    )
    .trigger(availableNow=True)
    .start(
        "abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_active_tourists"
    )
)

In [0]:
missing_query = (
    missing_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/gold_missing_tourists"
    )
    .trigger(availableNow=True)
    .start(
        "abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_missing_tourists"
    )
)

In [0]:
active_query.awaitTermination()
missing_query.awaitTermination()

In [0]:
ref_act_df = spark.read.format("delta").load("abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_missing_tourists")
ref_inact_df = spark.read.format("delta").load("abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_active_tourists")
display(ref_act_df)
print(ref_act_df.count())
display(ref_inact_df)
print(ref_inact_df.count())

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.gold_active_tourists
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_active_tourists';

CREATE TABLE IF NOT EXISTS default.gold_missing_tourists
USING DELTA
LOCATION 'abfss://gold@travelappprojectstorage.dfs.core.windows.net/tourist_monitoring/gold_missing_tourists';